In [862]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

## Overview

This notebook performs the full data cleaning and feature selection pipeline for the **Ames Housing Prices** dataset.

The pipeline follows a consistent **analysis → decision → apply** pattern across all variable types:
1. Inspect value distributions and correlations with the target (`SALEPRICE`)
2. Drop low-impact or redundant features
3. Encode remaining features appropriately

The variable types treated are:
- **Qualitative Nominal** — categories without order (e.g. foundation type, garage type)
- **Qualitative Ordinal** — categories with order (e.g. overall quality, kitchen quality)
- **Quantitative Discrete** — integer counts (e.g. number of rooms, year built)
- **Quantitative Continuous** — real-valued measurements (e.g. area, frontage)

The output is a cleaned `df_clean` dataframe ready for modelling.

In [863]:
# LOADING THE DATAFRAME

data_file = Path('..') / '01_data' / 'train.csv'
df_raw = pd.read_csv(data_file, sep=',')


In [864]:
df = df_raw.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [865]:
# DATA SET PREVIEW

pd.set_option('display.max_columns', None)
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## First Treatments

### Treatment of String Values

In [866]:
# CONVERT OBJECT TO STRING

columns_object = df.select_dtypes(include='object')
columns_str = columns_object.convert_dtypes(convert_string=True)

df[columns_object.columns] = columns_str

In [867]:
# STRING NORMALIZATION

df.columns = df.columns.str.upper()
columns_str = df.select_dtypes(include='string')

for col in columns_str.columns:
    df[col] = df[col].str.upper()

### Treatment of Null Values

In [868]:
null_values = df.isnull().mean()*100
null_values[null_values > 0].sort_values(ascending=False).round(2)

POOLQC          99.52
MISCFEATURE     96.30
ALLEY           93.77
FENCE           80.75
MASVNRTYPE      59.73
FIREPLACEQU     47.26
LOTFRONTAGE     17.74
GARAGETYPE       5.55
GARAGEYRBLT      5.55
GARAGEFINISH     5.55
GARAGEQUAL       5.55
GARAGECOND       5.55
BSMTEXPOSURE     2.60
BSMTFINTYPE2     2.60
BSMTQUAL         2.53
BSMTCOND         2.53
BSMTFINTYPE1     2.53
MASVNRAREA       0.55
ELECTRICAL       0.07
dtype: float64

In [869]:
high_missing_cols = null_values[null_values > 40]
medium_missing_cols = null_values[(null_values < 40) & (null_values > 10)]
low_missing_cols = null_values[null_values < 10]

df = df.drop(high_missing_cols.index, axis=1)
df["GARAGEYRBLT"] = df["GARAGEYRBLT"].fillna(0)

df = df.dropna(subset=low_missing_cols.index)

In [870]:
# KNN Method

imputer = KNNImputer(n_neighbors=5, weights='distance')

df[medium_missing_cols.index] = imputer.fit_transform(df[medium_missing_cols.index])

In [871]:
# There is a better way to do that, but it's more complicate, but the key-ideas are below:

# 1. Take a column where you KNOW all values
# 2. Artificially "hide" some of them
# 3. Impute with different n_neighbors
# 4. Compare imputed vs real values

## Classification of Variables

In [872]:
# Feature Engineering
df['TOTALLIVINGSF'] = df['TOTALBSMTSF'] + df['1STFLRSF'] + df['2NDFLRSF']
df['HOUSE_AGE'] = df['YRSOLD'] - df['YEARBUILT']
df['REMODEL_AGE'] = df['YRSOLD'] - df['YEARREMODADD']
df['GARAGE_PROXY'] = df['GARAGECARS'] * df['GARAGEAREA']
df['BATHROOM_INDEX'] = df['FULLBATH'] + 0.5*df['HALFBATH'] + 0.5*df['BSMTFULLBATH']
df['HASBASEMENT'] = (df['TOTALBSMTSF'] > 0).astype(int)
df['HASGARAGE'] = (df['GARAGEAREA'] > 0).astype(int)
df['HASFIREPLACE'] = (df['FIREPLACES'] > 0).astype(int)
df['HASPOOL'] = (df['POOLAREA'] > 0).astype(int)
df['TOTALPORCHSF'] = df['OPENPORCHSF'] + df['SCREENPORCH'] + df['3SSNPORCH']

# Drop used columns

cols_to_drop = [
    'TOTALBSMTSF', '1STFLRSF', '2NDFLRSF',
    'YEARBUILT', 'YEARREMODADD',
    'GARAGECARS', 'GARAGEAREA',
    'FULLBATH', 'HALFBATH', 'BSMTFULLBATH',
    'FIREPLACES', 'POOLAREA',
    'OPENPORCHSF', 'SCREENPORCH', '3SSNPORCH',
]

df.drop(columns=cols_to_drop, inplace=True)

In [873]:
columns_qual_nom = [
    "MSSUBCLASS",
    "MSZONING",
    "STREET",
    "LANDCONTOUR",
    "UTILITIES",
    "LOTCONFIG",
    "NEIGHBORHOOD",
    "CONDITION1",
    "CONDITION2",
    "BLDGTYPE",
    "HOUSESTYLE",
    "ROOFSTYLE",
    "ROOFMATL",
    "EXTERIOR1ST",
    "EXTERIOR2ND",
    "FOUNDATION",
    "HEATING",
    "CENTRALAIR",
    "ELECTRICAL",
    "GARAGETYPE",
    "GARAGEFINISH",
    "PAVEDDRIVE",
    "SALETYPE",
    "SALECONDITION",
]

columns_qual_ord = [
    "LOTSHAPE",
    "OVERALLQUAL",
    "OVERALLCOND",
    "EXTERQUAL",
    "EXTERCOND",
    "BSMTQUAL",
    "BSMTCOND",
    "BSMTEXPOSURE",
    "BSMTFINTYPE1",
    "BSMTFINTYPE2",
    "HEATINGQC",
    "KITCHENQUAL",
    "FUNCTIONAL",
    "GARAGEQUAL",
    "GARAGECOND",
    "LANDSLOPE",
]

columns_quan_disc = [
    "YRSOLD",
    "MOSOLD",
    "BSMTHALFBATH",
    "BEDROOMABVGR",
    "KITCHENABVGR",
    "TOTRMSABVGRD",
    "GARAGEYRBLT",
    "HASBASEMENT",
    "HASGARAGE",
    "HASFIREPLACE",
    "HASPOOL",
]

columns_quan_cont = [
    "LOTFRONTAGE",
    "LOTAREA",
    "MASVNRAREA",
    "BSMTFINSF1",
    "BSMTFINSF2",
    "BSMTUNFSF",
    "LOWQUALFINSF",
    "GRLIVAREA",
    "WOODDECKSF",
    "ENCLOSEDPORCH",
    "MISCVAL",
    "TOTALLIVINGSF",
    "HOUSE_AGE",
    "REMODEL_AGE",
    "GARAGE_PROXY",
    "BATHROOM_INDEX",
    "TOTALPORCHSF",
]

target = ["SALEPRICE"]


## Treatment of Qualitative Nominal Variables

### First Analysis

In [874]:
# FIRST ANALYSIS

df_clean_qual_nom = df.copy()

for col in columns_qual_nom:
    print(df_clean_qual_nom[col].value_counts())
    print()

MSSUBCLASS
20     502
60     294
50     129
120     86
160     61
80      57
70      57
30      51
90      28
190     21
85      19
75      14
45       9
180      6
40       4
Name: count, dtype: int64

MSZONING
RL         1066
RM          191
FV           62
RH           11
C (ALL)       8
Name: count, dtype: Int64

STREET
PAVE    1333
GRVL       5
Name: count, dtype: Int64

LANDCONTOUR
LVL    1206
BNK      52
HLS      48
LOW      32
Name: count, dtype: Int64

UTILITIES
ALLPUB    1337
NOSEWA       1
Name: count, dtype: Int64

LOTCONFIG
INSIDE     957
CORNER     244
CULDSAC     90
FR2         43
FR3          4
Name: count, dtype: Int64

NEIGHBORHOOD
NAMES      209
COLLGCR    146
OLDTOWN    100
SOMERST     83
GILBERT     77
NRIDGHT     75
NWAMES      73
EDWARDS     70
SAWYER      69
SAWYERW     53
CRAWFOR     50
BRKSIDE     47
MITCHEL     42
NORIDGE     41
TIMBER      37
IDOTRR      29
CLEARCR     26
STONEBR     25
SWISU       20
BLMNGTN     17
BRDALE      15
MEADOWV     12
VEENKER     

In [875]:
columns_to_drop = [
    'MSSUBCLASS',
    'STREET',
    'LANDCONTOUR',
    'NEIGHBORHOOD',
    'UTILITIES',
    'CONDITION2',
    'ROOFMATL',
    'EXTERIOR2ND',
    'HEATING',
    'CENTRALAIR',
    'ELECTRICAL',
    'PAVEDDRIVE',
]

df_clean_qual_nom = df_clean_qual_nom.drop(columns=columns_to_drop, axis=1)

# MSZONING
df_clean_qual_nom['MSZONING'] = df_clean_qual_nom[df_clean_qual_nom['MSZONING'] != 'C (ALL)']['MSZONING']

# LOTCONFIG
mapping_lotconfig = {
    'INSIDE': 'SINGLE_FRONTAGE',
    'CULDSAC': 'SINGLE_FRONTAGE',
    'CORNER': 'MULTIPLE_FRONTAGE',
    'FR2': 'MULTIPLE_FRONTAGE',
    'FR3': 'MULTIPLE_FRONTAGE',
}
df_clean_qual_nom['LOTCONFIG'] = df_clean_qual_nom['LOTCONFIG'].replace(
    mapping_lotconfig
)

# CONDITION1 - NOISE
df_clean_qual_nom = df_clean_qual_nom.rename(columns={'CONDITION1': 'NOISE'})

mapping_noise = {
    'ARTERY': 'HIGH',
    'FEEDR': 'HIGH',
    'RRNN': 'HIGH',
    'RRAN': 'HIGH',
    'RRNE': 'HIGH',
    'RRAE': 'HIGH',
    'POSN': 'LOW',
    'POSA': 'LOW',
    'NORM': 'NORMAL',
}
df_clean_qual_nom['NOISE'] = df_clean_qual_nom['NOISE'].replace(mapping_noise)

# BLDGTYPE
mapping_bldgtype = {
    'TWNHSE': 'TOWNHOUSE',
    'TWNHSI': 'TOWNHOUSE',
    'TWNHS': 'TOWNHOUSE',
    'DUPLEX': 'MULTI-FAMILY',
    '2FMCON': 'MULTI-FAMILY',
}
df_clean_qual_nom['BLDGTYPE'] = df_clean_qual_nom['BLDGTYPE'].replace(mapping_bldgtype)

# HOUSESTYLE
mapping_housestyle = {
    '1STORY': '1STORY',
    '2STORY': '2STORY',
    '1.5FIN': '1.5STORY',
    '1.5UNF': '1.5STORY',
    '2.5FIN': '2STORY',
    '2.5UNF': '2STORY',
    'SLVL': 'SPLIT',
    'SFOYER': 'SPLIT',
}
df_clean_qual_nom['HOUSESTYLE'] = df_clean_qual_nom['HOUSESTYLE'].replace(
    mapping_housestyle
)

# ROOFSTYLE
mapping_roofstyle = {
    'GABLE': 'GABLE',
    'HIP': 'NON-GABLE',
    'FLAT': 'NON-GABLE',
    'GAMBREL': 'NON-GABLE',
    'MANSARD': 'NON-GABLE',
    'SHED': 'NON-GABLE',
}
df_clean_qual_nom['ROOFSTYLE'] = df_clean_qual_nom['ROOFSTYLE'].replace(
    mapping_roofstyle
)

# EXTERIOR1ST
mapping_exterior1st = {
    'VINYLSD': 'VINYLSD',
    'HDBOARD': 'HDBOARD',
    'METALSD': 'METALSD',
    'WD SDNG': 'WD SDNG',
    'PLYWOOD': 'PLYWOOD',
    'CEMNTBD': 'OTHER',
    'BRKFACE': 'OTHER',
    'STUCCO': 'OTHER',
    'WDSHING': 'OTHER',
    'ASBSHNG': 'OTHER',
    'STONE': 'OTHER',
    'BRKCOMM': 'OTHER',
    'IMSTUCC': 'OTHER',
    'CBLOCK': 'OTHER',
}
df_clean_qual_nom['EXTERIOR1ST'] = df_clean_qual_nom['EXTERIOR1ST'].replace(
    mapping_exterior1st
)

# FOUNDATION
df_clean_qual_nom = df_clean_qual_nom[
    df_clean_qual_nom['FOUNDATION'].isin(['PCONC', 'CBLOCK', 'BRKTIL'])
]

# GARAGETYPE
mapping_garagetype = {
    'ATTCHD': 'ATTCHD',
    'DETCHD': 'DETCHD',
    'BUILTIN': 'OTHER',
    'BASMENT': 'OTHER',
    'CARPORT': 'OTHER',
    '2TYPES': 'OTHER',
}
df_clean_qual_nom['GARAGETYPE'] = df_clean_qual_nom['GARAGETYPE'].replace(
    mapping_garagetype
)

# SALETYPE
mapping_saletype = {
    'WD': 'WD',
    'NEW': 'NEW',
    'COD': 'COD',
    'CONLD': 'CON',
    'CONLI': 'CON',
    'CWD': 'WD',
    'CONLW': 'CON',
    'OTH': 'OTH',
}
df_clean_qual_nom['SALETYPE'] = df_clean_qual_nom['SALETYPE'].replace(mapping_saletype)
df_clean_qual_nom = df_clean_qual_nom[df_clean_qual_nom['SALETYPE'] != 'OTH']

# SALECONDITION
df_clean_qual_nom = df_clean_qual_nom[
    df_clean_qual_nom['SALECONDITION'].isin(['NORMAL', 'PARTIAL', 'ABNORML'])
]
df_clean_qual_nom['SALECONDITION'] = df_clean_qual_nom['SALECONDITION'].replace(
    'ABNORML', 'ABNORMAL'
)

# FILTERING LIST
columns_qual_nom = list(
    (set(columns_qual_nom) - set(columns_to_drop)) - {'CONDITION1'} | {'NOISE'}
)


In [876]:
df_clean_qual_nom[columns_qual_nom].head()

,BLDGTYPE,GARAGEFINISH,EXTERIOR1ST,SALECONDITION,FOUNDATION,ROOFSTYLE,GARAGETYPE,SALETYPE,HOUSESTYLE,LOTCONFIG,MSZONING,NOISE
0,1FAM,RFN,VINYLSD,NORMAL,PCONC,GABLE,ATTCHD,WD,2STORY,SINGLE_FRONTAGE,RL,NORMAL
1,1FAM,RFN,METALSD,NORMAL,CBLOCK,GABLE,ATTCHD,WD,1STORY,MULTIPLE_FRONTAGE,RL,HIGH
2,1FAM,RFN,VINYLSD,NORMAL,PCONC,GABLE,ATTCHD,WD,2STORY,SINGLE_FRONTAGE,RL,NORMAL
3,1FAM,UNF,WD SDNG,ABNORMAL,BRKTIL,GABLE,DETCHD,WD,2STORY,MULTIPLE_FRONTAGE,RL,NORMAL
4,1FAM,RFN,VINYLSD,NORMAL,PCONC,GABLE,ATTCHD,WD,2STORY,MULTIPLE_FRONTAGE,RL,NORMAL


In [877]:
# SECOND ANALYSIS

for col in columns_qual_nom:
    print(df_clean_qual_nom[col].value_counts())
    print()

BLDGTYPE
1FAM            1109
TOWNHOUSE        148
MULTI-FAMILY      43
Name: count, dtype: Int64

GARAGEFINISH
UNF    556
RFN    406
FIN    338
Name: count, dtype: Int64

EXTERIOR1ST
VINYLSD    480
HDBOARD    208
METALSD    194
WD SDNG    176
OTHER      150
PLYWOOD     92
Name: count, dtype: Int64

SALECONDITION
NORMAL      1096
PARTIAL      120
ABNORMAL      84
Name: count, dtype: Int64

FOUNDATION
PCONC     612
CBLOCK    564
BRKTIL    124
Name: count, dtype: Int64

ROOFSTYLE
GABLE        1006
NON-GABLE     294
Name: count, dtype: Int64

GARAGETYPE
ATTCHD    833
DETCHD    356
OTHER     111
Name: count, dtype: Int64

SALETYPE
WD     1125
NEW     117
COD      42
CON      16
Name: count, dtype: Int64

HOUSESTYLE
1STORY      641
2STORY      432
1.5STORY    138
SPLIT        89
Name: count, dtype: Int64

LOTCONFIG
SINGLE_FRONTAGE      1019
MULTIPLE_FRONTAGE     281
Name: count, dtype: Int64

MSZONING
RL    1041
RM     180
FV      62
RH      11
Name: count, dtype: Int64

NOISE
NORMAL    112

### One-Hot Encoding

Columns with no natural ordering are encoded with `pd.get_dummies`. A temporary dummy dataframe is used to measure the summed absolute correlation with `SALEPRICE` before deciding which columns to keep.

In [878]:
columns_to_dummy_analysis = [
    'FOUNDATION',
    'EXTERIOR1ST',
    'GARAGETYPE',
    'BLDGTYPE',
    'LOTCONFIG',
    'ROOFSTYLE',
    'SALETYPE'
]

df_clean_qual_nom_dummy = df_clean_qual_nom[columns_to_dummy_analysis].copy()

for col in columns_to_dummy_analysis:
    df_clean_qual_nom_dummy = pd.get_dummies(df_clean_qual_nom_dummy, columns=[col], drop_first=False, dtype=int)

df_clean_qual_nom_dummy[target] = df_clean_qual_nom[target]
df_clean_qual_nom_dummy.head()

,FOUNDATION_BRKTIL,FOUNDATION_CBLOCK,FOUNDATION_PCONC,EXTERIOR1ST_HDBOARD,EXTERIOR1ST_METALSD,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_ATTCHD,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,BLDGTYPE_1FAM,BLDGTYPE_MULTI-FAMILY,BLDGTYPE_TOWNHOUSE,LOTCONFIG_MULTIPLE_FRONTAGE,LOTCONFIG_SINGLE_FRONTAGE,ROOFSTYLE_GABLE,ROOFSTYLE_NON-GABLE,SALETYPE_COD,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD,SALEPRICE
0,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,208500
1,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,181500
2,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,223500
3,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,1,0,1,0,0,0,0,1,140000
4,0,0,1,0,0,0,0,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,250000


In [879]:
df_clean_qual_nom_dummy_occor = df_clean_qual_nom_dummy.corr()
df_clean_qual_nom_dummy_occor = df_clean_qual_nom_dummy_occor["SALEPRICE"].abs()

df_clean_qual_nom_dummy_occor.index = df_clean_qual_nom_dummy_occor.index.str.split("_", n =0).str[0]
df_clean_qual_nom_dummy_occor = df_clean_qual_nom_dummy_occor.groupby(by=df_clean_qual_nom_dummy_occor.index).sum()

df_clean_qual_nom_dummy_occor.sort_values(ascending=False)

FOUNDATION     1.055134
SALEPRICE      1.000000
GARAGETYPE     0.846281
EXTERIOR1ST    0.836259
SALETYPE       0.705247
ROOFSTYLE      0.465324
BLDGTYPE       0.311500
LOTCONFIG      0.000571
Name: SALEPRICE, dtype: float64

> **Decision:** `ROOFSTYLE`, `BLDGTYPE`, and `LOTCONFIG` show negligible summed correlation with `SALEPRICE` and are dropped.
> The remaining nominal columns (`FOUNDATION`, `EXTERIOR1ST`, `GARAGETYPE`, `SALETYPE`) are encoded as dummies.

In [880]:
columns_to_drop = [
    'ROOFSTYLE',
    'BLDGTYPE',
    'LOTCONFIG'
]


# FILTERING LIST
columns_qual_nom = list(
    (set(columns_qual_nom) - set(columns_to_drop))
)


df_clean_qual_nom = df_clean_qual_nom.drop(columns=columns_to_drop, axis=1)

columns_to_dummy = [
    'FOUNDATION',
    'EXTERIOR1ST',
    'GARAGETYPE',
    'SALETYPE'
]

for col in columns_to_dummy:
    df_clean_qual_nom = pd.get_dummies(df_clean_qual_nom, columns=[col], drop_first=True, dtype="uint8")



### Label Encoding

Columns that carry implicit ordinal meaning (e.g. `GARAGEFINISH`, `NOISE`) are encoded with a hand-crafted ordinal mapping. A temporary frame is used to check correlation before deciding which columns to keep.

In [881]:
columns_qual_nom_encoder = list(
    (set(columns_qual_nom) - set(columns_to_dummy))
)

for col in columns_qual_nom_encoder:
    print(df_clean_qual_nom[col].value_counts())
    print()

GARAGEFINISH
UNF    556
RFN    406
FIN    338
Name: count, dtype: Int64

SALECONDITION
NORMAL      1096
PARTIAL      120
ABNORMAL      84
Name: count, dtype: Int64

HOUSESTYLE
1STORY      641
2STORY      432
1.5STORY    138
SPLIT        89
Name: count, dtype: Int64

MSZONING
RL    1041
RM     180
FV      62
RH      11
Name: count, dtype: Int64

NOISE
NORMAL    1128
HIGH       145
LOW         27
Name: count, dtype: Int64



In [882]:
columns_to_le_analysis = columns_qual_nom_encoder
df_clean_qual_nom_le = df_clean_qual_nom[columns_to_le_analysis].copy()

In [883]:
# NOISE
df_clean_qual_nom_le["NOISE"] = df_clean_qual_nom_le["NOISE"].map(
    {"LOW": 0, "NORMAL": 1, "HIGH": 2}
)

# MSZONING
df_clean_qual_nom_le["MSZONING"] = df_clean_qual_nom_le["MSZONING"].map(
    {"RH": 0, "FV": 1, "RM": 2, "RL": 3}
)

# SALECONDITION
df_clean_qual_nom_le["SALECONDITION"] = df_clean_qual_nom_le["SALECONDITION"].map(
    {"ABNORMAL": 0, "PARTIAL": 1, "NORMAL": 2}
)

# HOUSESTYLE
df_clean_qual_nom_le["HOUSESTYLE"] = df_clean_qual_nom_le["HOUSESTYLE"].map(
    {"1STORY": 0, "1.5STORY": 1, "SPLIT": 2, "2STORY": 3}
)

# GARAGEFINISH
df_clean_qual_nom_le["GARAGEFINISH"] = df_clean_qual_nom_le["GARAGEFINISH"].map(
    {"UNF": 0, "RFN": 1, "FIN": 2}
)

In [884]:
df_clean_qual_nom_le[target] = df_clean_qual_nom[target]
df_clean_qual_nom_le.head()

,GARAGEFINISH,SALECONDITION,HOUSESTYLE,MSZONING,NOISE,SALEPRICE
0,1,2,3,3.0,1,208500
1,1,2,0,3.0,2,181500
2,1,2,3,3.0,1,223500
3,0,0,3,3.0,1,140000
4,1,2,3,3.0,1,250000


In [885]:
df_clean_qual_nom_le_ocorr = df_clean_qual_nom_le.corr()
df_clean_qual_nom_le_ocorr["SALEPRICE"].sort_values(ascending=False).round(2).abs()

SALEPRICE        1.00
GARAGEFINISH     0.51
HOUSESTYLE       0.15
MSZONING         0.15
SALECONDITION    0.09
NOISE            0.15
Name: SALEPRICE, dtype: float64

In [886]:
columns_to_drop = [
    'HOUSESTYLE',
    'MSZONING',
    'SALECONDITION',
    'NOISE'
]

df_clean_qual_nom = df_clean_qual_nom.drop(columns=columns_to_drop, axis=1)

# GARAGEFINISH
mapping_garagefinish = {
    'FIN': 2,
    'RFN': 1,
    'UNF': 0
}

df_clean_qual_nom['GARAGEFINISH'] = df_clean_qual_nom['GARAGEFINISH'].map(mapping_garagefinish)

In [887]:
df_clean_qual_nom.head()

,ID,LOTFRONTAGE,LOTAREA,LOTSHAPE,LANDSLOPE,OVERALLQUAL,OVERALLCOND,MASVNRAREA,EXTERQUAL,EXTERCOND,BSMTQUAL,BSMTCOND,BSMTEXPOSURE,BSMTFINTYPE1,BSMTFINSF1,BSMTFINTYPE2,BSMTFINSF2,BSMTUNFSF,HEATINGQC,LOWQUALFINSF,GRLIVAREA,BSMTHALFBATH,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,FUNCTIONAL,GARAGEYRBLT,GARAGEFINISH,GARAGEQUAL,GARAGECOND,WOODDECKSF,ENCLOSEDPORCH,MISCVAL,MOSOLD,YRSOLD,SALEPRICE,TOTALLIVINGSF,HOUSE_AGE,REMODEL_AGE,GARAGE_PROXY,BATHROOM_INDEX,HASBASEMENT,HASGARAGE,HASFIREPLACE,HASPOOL,TOTALPORCHSF,FOUNDATION_CBLOCK,FOUNDATION_PCONC,EXTERIOR1ST_METALSD,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD
0,1,65.0,8450,REG,GTL,7,5,196.0,GD,TA,GD,TA,NO,GLQ,706,UNF,0,150,EX,0,1710,0,3,1,GD,8,TYP,2003.0,1,TA,TA,0,0,0,2,2008,208500,2566,5,5,1096,3.0,1,1,0,0,61,0,1,0,0,0,1,0,0,0,0,0,1
1,2,80.0,9600,REG,GTL,6,8,0.0,TA,TA,GD,TA,GD,ALQ,978,UNF,0,284,EX,0,1262,1,3,1,TA,6,TYP,1976.0,1,TA,TA,298,0,0,5,2007,181500,2524,31,31,920,2.0,1,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0,1
2,3,68.0,11250,IR1,GTL,7,5,162.0,GD,TA,GD,TA,MN,GLQ,486,UNF,0,434,EX,0,1786,0,3,1,GD,6,TYP,2001.0,1,TA,TA,0,0,0,9,2008,223500,2706,7,6,1216,3.0,1,1,1,0,42,0,1,0,0,0,1,0,0,0,0,0,1
3,4,60.0,9550,IR1,GTL,7,5,0.0,TA,TA,TA,GD,NO,ALQ,216,UNF,0,540,GD,0,1717,0,3,1,GD,7,TYP,1998.0,0,TA,TA,0,272,0,2,2006,140000,2473,91,36,1926,1.5,1,1,1,0,35,0,0,0,0,0,0,1,1,0,0,0,1
4,5,84.0,14260,IR1,GTL,8,5,350.0,GD,TA,GD,TA,AV,GLQ,655,UNF,0,490,EX,0,2198,0,4,1,GD,9,TYP,2000.0,1,TA,TA,192,0,0,12,2008,250000,3343,8,8,2508,3.0,1,1,1,0,84,0,1,0,0,0,1,0,0,0,0,0,1


## Treatment of Qualitative Ordinal Variables

In [888]:
df_clean_qual_ord = df_clean_qual_nom.copy()


for col in columns_qual_ord:
    print(df_clean_qual_ord[col].value_counts().sort_values(ascending=False))
    print()

LOTSHAPE
REG    801
IR1    451
IR2     39
IR3      9
Name: count, dtype: Int64

OVERALLQUAL
6     353
5     338
7     302
8     162
4      76
9      43
10     17
3       7
2       2
Name: count, dtype: int64

OVERALLCOND
5    751
6    229
7    175
8     66
4     42
9     19
3     15
2      3
Name: count, dtype: int64

EXTERQUAL
TA    774
GD    470
EX     50
FA      6
Name: count, dtype: Int64

EXTERCOND
TA    1152
GD     131
FA      15
EX       2
Name: count, dtype: Int64

BSMTQUAL
GD    578
TA    572
EX    120
FA     30
Name: count, dtype: Int64

BSMTCOND
TA    1202
GD      60
FA      37
PO       1
Name: count, dtype: Int64

BSMTEXPOSURE
NO    863
AV    205
GD    123
MN    109
Name: count, dtype: Int64

BSMTFINTYPE1
GLQ    393
UNF    374
ALQ    206
BLQ    140
REC    121
LWQ     66
Name: count, dtype: Int64

BSMTFINTYPE2
UNF    1140
REC      53
LWQ      44
BLQ      32
ALQ      19
GLQ      12
Name: count, dtype: Int64

HEATINGQC
EX    693
TA    361
GD    211
FA     34
PO      1
Name: co

In [889]:
df_clean_qual_ord_analysis = df_clean_qual_ord.copy()
df_clean_qual_ord_analysis = df_clean_qual_ord_analysis[columns_qual_ord]

In [890]:
# OVERALLQUAL (already done)
# OVERALLCOND (already done)

# LOTSHAPE
mapping_lotshape = {
    'IR3': 0,
    'IR2': 1,
    'IR1': 2,
    'REG': 3
}

df_clean_qual_ord_analysis['LOTSHAPE'] = df_clean_qual_ord_analysis['LOTSHAPE'].map(mapping_lotshape)


# Shared mappings
mapping_qual = {
    'PO': 0,
    'FA': 1,
    'TA': 2,
    'GD': 3,
    'EX': 4
}

# EXTERQUAL
df_clean_qual_ord_analysis['EXTERQUAL'] = df_clean_qual_ord_analysis['EXTERQUAL'].map(mapping_qual)

# EXTERCOND
df_clean_qual_ord_analysis['EXTERCOND'] = df_clean_qual_ord_analysis['EXTERCOND'].map(mapping_qual)

# HEATINGQC
df_clean_qual_ord_analysis['HEATINGQC'] = df_clean_qual_ord_analysis['HEATINGQC'].map(mapping_qual)

# KITCHENQUAL
df_clean_qual_ord_analysis['KITCHENQUAL'] = df_clean_qual_ord_analysis['KITCHENQUAL'].map(mapping_qual)

# GARAGEQUAL
df_clean_qual_ord_analysis['GARAGEQUAL'] = df_clean_qual_ord_analysis['GARAGEQUAL'].map(mapping_qual)

# GARAGECOND
df_clean_qual_ord_analysis['GARAGECOND'] = df_clean_qual_ord_analysis['GARAGECOND'].map(mapping_qual)



# BSMTQUAL
mapping_bsmt = {
    'NA': 0,
    'PO': 1,
    'FA': 2,
    'TA': 3,
    'GD': 4,
    'EX': 5
}

df_clean_qual_ord_analysis['BSMTQUAL'] = df_clean_qual_ord_analysis['BSMTQUAL'].map(mapping_bsmt)

# BSMTCOND
df_clean_qual_ord_analysis['BSMTCOND'] = df_clean_qual_ord_analysis['BSMTCOND'].map(mapping_bsmt)


## BSMTEXPOSURE
mapping_bsmtexposure = {
    'NA': 0,
    'NO': 1,
    'MN': 2,
    'AV': 3,
    'GD': 4
}

df_clean_qual_ord_analysis['BSMTEXPOSURE'] = df_clean_qual_ord_analysis['BSMTEXPOSURE'].map(mapping_bsmtexposure)


# BSMTFINTYPE1
mapping_bsmtfintype = {
    'NA': 0,
    'UNF': 1,
    'LWQ': 2,
    'REC': 3,
    'BLQ': 4,
    'ALQ': 5,
    'GLQ': 6
}

df_clean_qual_ord_analysis['BSMTFINTYPE1'] = df_clean_qual_ord_analysis['BSMTFINTYPE1'].map(mapping_bsmtfintype)

# BSMTFINTYPE2
df_clean_qual_ord_analysis['BSMTFINTYPE2'] = df_clean_qual_ord_analysis['BSMTFINTYPE2'].map(mapping_bsmtfintype)


# FUNCTIONAL
mapping_functional = {
    'SAL': 0,
    'SEV': 1,
    'MAJ2': 2,
    'MAJ1': 3,
    'MOD': 4,
    'MIN2': 5,
    'MIN1': 6,
    'TYP': 7
}

df_clean_qual_ord_analysis['FUNCTIONAL'] = df_clean_qual_ord_analysis['FUNCTIONAL'].map(mapping_functional)

# LANDSLOPE
mapping_landslope = {
    'SEV': 0,
    'MOD': 1,
    'GTL': 2
}

df_clean_qual_ord_analysis['LANDSLOPE'] = df_clean_qual_ord_analysis['LANDSLOPE'].map(mapping_landslope)

In [891]:
df_clean_qual_ord_analysis[columns_qual_ord].isnull().sum()

LOTSHAPE        0
OVERALLQUAL     0
OVERALLCOND     0
EXTERQUAL       0
EXTERCOND       0
BSMTQUAL        0
BSMTCOND        0
BSMTEXPOSURE    0
BSMTFINTYPE1    0
BSMTFINTYPE2    0
HEATINGQC       0
KITCHENQUAL     0
FUNCTIONAL      0
GARAGEQUAL      0
GARAGECOND      0
LANDSLOPE       0
dtype: int64

In [892]:
df_clean_qual_ord_analysis[target] = df_clean_qual_ord[target]

In [893]:
df_clean_qual_ord_analysis_corr = df_clean_qual_ord_analysis.corr()
df_clean_qual_ord_analysis_corr['SALEPRICE'].abs().sort_values(ascending=False)

SALEPRICE       1.000000
OVERALLQUAL     0.785546
EXTERQUAL       0.670238
KITCHENQUAL     0.651949
BSMTQUAL        0.644768
HEATINGQC       0.415459
BSMTEXPOSURE    0.364113
BSMTFINTYPE1    0.262152
LOTSHAPE        0.255858
GARAGEQUAL      0.157884
BSMTCOND        0.143478
GARAGECOND      0.119022
OVERALLCOND     0.118040
FUNCTIONAL      0.092440
LANDSLOPE       0.067042
BSMTFINTYPE2    0.050797
EXTERCOND       0.019142
Name: SALEPRICE, dtype: float64

In [ ]:
mask = df_clean_qual_ord_analysis_corr < 0.4
columns_to_drop = mask[mask].index
df_clean_quan_disc = df_clean_qual_ord.drop(columns=columns_to_drop, axis=1)

In [894]:
columns_to_drop = [
    'BSMTEXPOSURE',
    'BSMTFINTYPE1',
    'LOTSHAPE',
    'GARAGEQUAL',
    'BSMTCOND',
    'GARAGECOND',
    'OVERALLCOND',
    'FUNCTIONAL',
    'LANDSLOPE',
    'BSMTFINTYPE2',
    'EXTERCOND'
]

df_clean_qual_ord = df_clean_qual_ord.drop(columns=columns_to_drop, axis=1)

# Shared mappings
mapping_qual = {
    'PO': 0,
    'FA': 1,
    'TA': 2,
    'GD': 3,
    'EX': 4
}

# EXTERQUAL
df_clean_qual_ord['EXTERQUAL'] = df_clean_qual_ord['EXTERQUAL'].map(mapping_qual)

# KITCHENQUAL
df_clean_qual_ord['KITCHENQUAL'] = df_clean_qual_ord['KITCHENQUAL'].map(mapping_qual)

# HEATINGQC
df_clean_qual_ord['HEATINGQC'] = df_clean_qual_ord['HEATINGQC'].map(mapping_qual)

# BSMTQUAL
mapping_bsmt = {
    'NA': 0,
    'PO': 1,
    'FA': 2,
    'TA': 3,
    'GD': 4,
    'EX': 5
}

df_clean_qual_ord['BSMTQUAL'] = df_clean_qual_ord['BSMTQUAL'].map(mapping_bsmt)


In [895]:
df_clean_qual_ord.head()

,ID,LOTFRONTAGE,LOTAREA,OVERALLQUAL,MASVNRAREA,EXTERQUAL,BSMTQUAL,BSMTFINSF1,BSMTFINSF2,BSMTUNFSF,HEATINGQC,LOWQUALFINSF,GRLIVAREA,BSMTHALFBATH,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,GARAGEYRBLT,GARAGEFINISH,WOODDECKSF,ENCLOSEDPORCH,MISCVAL,MOSOLD,YRSOLD,SALEPRICE,TOTALLIVINGSF,HOUSE_AGE,REMODEL_AGE,GARAGE_PROXY,BATHROOM_INDEX,HASBASEMENT,HASGARAGE,HASFIREPLACE,HASPOOL,TOTALPORCHSF,FOUNDATION_CBLOCK,FOUNDATION_PCONC,EXTERIOR1ST_METALSD,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD
0,1,65.0,8450,7,196.0,3,4,706,0,150,4,0,1710,0,3,1,3,8,2003.0,1,0,0,0,2,2008,208500,2566,5,5,1096,3.0,1,1,0,0,61,0,1,0,0,0,1,0,0,0,0,0,1
1,2,80.0,9600,6,0.0,2,4,978,0,284,4,0,1262,1,3,1,2,6,1976.0,1,298,0,0,5,2007,181500,2524,31,31,920,2.0,1,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0,1
2,3,68.0,11250,7,162.0,3,4,486,0,434,4,0,1786,0,3,1,3,6,2001.0,1,0,0,0,9,2008,223500,2706,7,6,1216,3.0,1,1,1,0,42,0,1,0,0,0,1,0,0,0,0,0,1
3,4,60.0,9550,7,0.0,2,3,216,0,540,3,0,1717,0,3,1,3,7,1998.0,0,0,272,0,2,2006,140000,2473,91,36,1926,1.5,1,1,1,0,35,0,0,0,0,0,0,1,1,0,0,0,1
4,5,84.0,14260,8,350.0,3,4,655,0,490,4,0,2198,0,4,1,3,9,2000.0,1,192,0,0,12,2008,250000,3343,8,8,2508,3.0,1,1,1,0,84,0,1,0,0,0,1,0,0,0,0,0,1


## Treatment of Quantitative Discrete Variables

> **Note:** Discrete variables show fewer extreme outliers than continuous ones. Given their low frequency, outlier rows for this variable type are retained and only low-correlation columns are dropped.

In [ ]:
df_clean_quan_disc_analysis = df_clean_qual_ord[columns_quan_disc].copy()

df_clean_quan_disc_analysis[target] = df_clean_qual_ord[target]

df_clean_quan_disc_analysis_corr = df_clean_quan_disc_analysis.corr()
df_clean_quan_disc_analysis_corr["SALEPRICE"] = df_clean_quan_disc_analysis_corr["SALEPRICE"].abs()
df_clean_quan_disc_analysis_corr = df_clean_quan_disc_analysis_corr["SALEPRICE"].sort_values(ascending=False)

In [897]:
mask = df_clean_quan_disc_analysis_corr < 0.4
columns_to_drop = mask[mask].index
df_clean_quan_disc = df_clean_qual_ord.drop(columns=columns_to_drop, axis=1)

## Treatment of Quantitative Continuous Variables

### Outlier Treatment

In [898]:
# SKEWNESS ANALYSIS

df_clean_quan_cont = df_clean_quan_disc.copy()

skewness = {}

for col in columns_quan_cont:
    skewness[col] = df_clean_quan_cont[col].skew()

df_skewness = pd.DataFrame.from_dict(skewness, orient="index", columns=["SKEWNESS"])
df_skewness = df_skewness.reset_index()
df_skewness = df_skewness.rename(columns={"index": "CATEGORY"})

df_skewness.sort_values(by=["SKEWNESS"], ascending=False).reset_index(drop=True)

columns_to_log = df_skewness[
    (df_skewness["SKEWNESS"] > 1) | (df_skewness["SKEWNESS"] < -1)
]
columns_normal = df_skewness[
    (df_skewness["SKEWNESS"] < 1) & (df_skewness["SKEWNESS"] > -1)
]


# LOG TRANSFORMATION
for col in columns_to_log["CATEGORY"]:
    df_clean_quan_cont[col] = np.log1p(df_clean_quan_cont[col])


# CLEANING OF OUTLIERS
prop_to_delete = 0.04

candidates = [
    (0.25, 0.75),
    (0.20, 0.80),
    (0.15, 0.85),
    (0.10, 0.90),
    (0.05, 0.95),
    (0.02, 0.98),
    (0.01, 0.99),
]

for col in columns_to_log["CATEGORY"]:
    n_total = len(df_clean_quan_cont[col])

    for q1_pct, q3_pct in candidates:
        q1 = df_clean_quan_cont[col].quantile(q1_pct)
        q3 = df_clean_quan_cont[col].quantile(q3_pct)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (df_clean_quan_cont[col] >= lower) & (df_clean_quan_cont[col] <= upper)
        pct_removed = 1 - mask.sum() / n_total

        if pct_removed < prop_to_delete:
            break

    # print(f"{col}: quantiles=({q1_pct}, {q3_pct}) | bounds=[{lower:.2f}, {upper:.2f}] | removed={pct_removed:.1%}")
    df_clean_quan_cont = df_clean_quan_cont[mask]

for col in columns_normal["CATEGORY"]:
    n_total = len(df_clean_quan_cont[col])

    for q1_pct, q3_pct in candidates:
        q1 = df_clean_quan_cont[col].quantile(q1_pct)
        q3 = df_clean_quan_cont[col].quantile(q3_pct)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (df_clean_quan_cont[col] >= lower) & (df_clean_quan_cont[col] <= upper)
        pct_removed = 1 - mask.sum() / n_total

        if pct_removed < prop_to_delete:
            break

    # print(f"{col}: quantiles=({q1_pct}, {q3_pct}) | bounds=[{lower:.2f}, {upper:.2f}] | removed={pct_removed:.1%}")
    df_clean_quan_cont = df_clean_quan_cont[mask]

print(
    100 - round(df_clean_quan_cont.shape[0] / df_raw.shape[0], 2) * 100,
    "% of data was deleted",
)


22.0 % of data was deleted


In [899]:
df_clean_quan_cont.head()

,ID,LOTFRONTAGE,LOTAREA,OVERALLQUAL,MASVNRAREA,EXTERQUAL,BSMTQUAL,BSMTFINSF1,BSMTFINSF2,BSMTUNFSF,HEATINGQC,LOWQUALFINSF,GRLIVAREA,KITCHENQUAL,TOTRMSABVGRD,GARAGEYRBLT,GARAGEFINISH,WOODDECKSF,ENCLOSEDPORCH,MISCVAL,SALEPRICE,TOTALLIVINGSF,HOUSE_AGE,REMODEL_AGE,GARAGE_PROXY,BATHROOM_INDEX,HASBASEMENT,HASGARAGE,HASFIREPLACE,TOTALPORCHSF,FOUNDATION_CBLOCK,FOUNDATION_PCONC,EXTERIOR1ST_METALSD,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD
0,1,4.189655,9.042040,7,5.283204,3,4,6.561031,0.0,150,4,0.0,7.444833,3,8,2003.0,1,0.000000,0.000000,0.0,208500,7.850493,5,5,7.000334,3.0,1,1,0,4.127134,0,1,0,0,0,1,0,0,0,0,0,1
1,2,4.394449,9.169623,6,0.000000,2,4,6.886532,0.0,284,4,0.0,7.141245,2,6,1976.0,1,5.700444,0.000000,0.0,181500,7.833996,31,31,6.825460,2.0,1,1,1,0.000000,1,0,1,0,0,0,0,0,0,0,0,1
2,3,4.234107,9.328212,7,5.093750,3,4,6.188264,0.0,434,4,0.0,7.488294,3,6,2001.0,1,0.000000,0.000000,0.0,223500,7.903596,7,6,7.104144,3.0,1,1,1,3.761200,0,1,0,0,0,1,0,0,0,0,0,1
3,4,4.110874,9.164401,7,0.000000,2,3,5.379897,0.0,540,3,0.0,7.448916,3,7,1998.0,0,0.000000,5.609472,0.0,140000,7.813592,91,36,7.563720,1.5,1,1,1,3.583519,0,0,0,0,0,0,1,1,0,0,0,1
4,5,4.442651,9.565284,8,5.860786,3,4,6.486161,0.0,490,4,0.0,7.695758,3,9,2000.0,1,5.262690,0.000000,0.0,250000,8.114923,8,8,7.827640,3.0,1,1,1,4.442651,0,1,0,0,0,1,0,0,0,0,0,1


In [900]:
df_clean_quan_cont_analysis = df_clean_quan_cont[columns_quan_cont].copy()

df_clean_quan_cont_analysis[target] = df_clean_qual_ord[target]

df_clean_quan_cont_analysis_corr = df_clean_quan_cont_analysis.corr()

df_clean_quan_cont_analysis_corr["SALEPRICE"].abs().sort_values(ascending=False)

SALEPRICE         1.000000
TOTALLIVINGSF     0.785937
GRLIVAREA         0.702556
BATHROOM_INDEX    0.679186
GARAGE_PROXY      0.637194
HOUSE_AGE         0.569295
REMODEL_AGE       0.530539
MASVNRAREA        0.430475
TOTALPORCHSF      0.410956
LOTAREA           0.324300
WOODDECKSF        0.302018
LOTFRONTAGE       0.278692
ENCLOSEDPORCH     0.192749
BSMTFINSF1        0.170464
BSMTFINSF2        0.112662
BSMTUNFSF         0.101457
LOWQUALFINSF           NaN
MISCVAL                NaN
Name: SALEPRICE, dtype: float64

In [901]:
columns_to_drop = [
    "LOTAREA",
    "WOODDECKSF",
    "2NDFLRSF",
    "LOTFRONTAGE",
    "ENCLOSEDPORCH",
    "BSMTFINSF1",
    "BSMTUNFSF",
    "BSMTFINSF2",
    "SCREENPORCH",
    "LOWQUALFINSF",
    "3SSNPORCH",
    "POOLAREA",
    "MISCVAL"
]


# FILTERING LIST
columns_quan_cont = list(
    (set(columns_quan_cont) - set(columns_to_drop))
)

columns_quan_cont
df_clean_quan_cont = df_clean_quan_cont.drop(columns=columns_to_drop, axis=1)
std = preprocessing.StandardScaler()
df_clean_quan_cont[columns_quan_cont] = std.fit_transform(df_clean_quan_cont[columns_quan_cont])


KeyError: "['2NDFLRSF', 'SCREENPORCH', '3SSNPORCH', 'POOLAREA'] not found in axis"

## Final Dataset

All cleaning and feature-selection steps have been applied. The `df_clean` dataframe below contains only the retained features plus the `SALEPRICE` target, ready for the modelling phase.

In [ ]:
df_clean = df_clean_quan_cont.copy()

df_clean = df_clean.drop(columns=['ID', 'SALEPRICE'], axis=1)

df_clean[target] = df_clean_quan_cont[target]

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1086 entries, 0 to 1459
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   OVERALLQUAL          1086 non-null   int64  
 1   YEARBUILT            1086 non-null   int64  
 2   YEARREMODADD         1086 non-null   int64  
 3   MASVNRAREA           1086 non-null   float64
 4   EXTERQUAL            1086 non-null   int64  
 5   BSMTQUAL             1086 non-null   int64  
 6   TOTALBSMTSF          1086 non-null   float64
 7   HEATINGQC            1086 non-null   int64  
 8   1STFLRSF             1086 non-null   float64
 9   GRLIVAREA            1086 non-null   float64
 10  FULLBATH             1086 non-null   int64  
 11  KITCHENQUAL          1086 non-null   int64  
 12  TOTRMSABVGRD         1086 non-null   int64  
 13  FIREPLACES           1086 non-null   int64  
 14  GARAGEYRBLT          1086 non-null   float64
 15  GARAGEFINISH         1086 non-null   int64 

In [ ]:
df_clean.head()

,OVERALLQUAL,YEARBUILT,YEARREMODADD,MASVNRAREA,EXTERQUAL,BSMTQUAL,TOTALBSMTSF,HEATINGQC,1STFLRSF,GRLIVAREA,FULLBATH,KITCHENQUAL,TOTRMSABVGRD,FIREPLACES,GARAGEYRBLT,GARAGEFINISH,GARAGECARS,GARAGEAREA,OPENPORCHSF,FOUNDATION_CBLOCK,FOUNDATION_PCONC,EXTERIOR1ST_METALSD,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD,SALEPRICE
0,7,2003,2003,1.179624,3,4,-0.644764,4,-0.953682,0.576734,2,3,8,0,2003.0,1,2,0.358751,0.812098,0,1,0,0,0,1,0,0,0,0,0,1,0.573156
1,6,1976,1976,-0.840151,2,4,0.651156,4,0.443834,-0.426417,2,2,6,1,1976.0,1,2,-0.174861,-1.125982,1,0,1,0,0,0,0,0,0,0,0,1,0.155107
2,7,2001,2002,1.107196,3,4,-0.404091,4,-0.694140,0.720340,2,3,6,1,2001.0,1,2,0.722577,0.640257,0,1,0,0,0,1,0,0,0,0,0,1,0.782574
3,7,1915,1970,-0.840151,2,3,-1.059379,3,-0.537186,0.590224,1,3,7,1,1998.0,0,3,0.928745,0.556819,0,0,0,0,0,0,1,1,0,0,0,1,-0.627472
4,8,2000,2000,1.400435,3,4,0.326306,4,0.093516,1.405868,2,3,9,1,2000.0,1,3,2.105116,0.960263,0,1,0,0,0,1,0,0,0,0,0,1,1.120338


In [ ]:
df_clean_corr = df_clean.corr()
df_clean_corr = df_clean_corr["SALEPRICE"].abs().round(2)

df_clean_corr.sort_values(ascending=False)

SALEPRICE              1.00
OVERALLQUAL            0.81
GRLIVAREA              0.72
EXTERQUAL              0.68
BSMTQUAL               0.67
KITCHENQUAL            0.66
GARAGECARS             0.65
YEARBUILT              0.63
GARAGEAREA             0.62
FULLBATH               0.61
YEARREMODADD           0.58
GARAGEYRBLT            0.57
GARAGEFINISH           0.57
FOUNDATION_PCONC       0.55
TOTALBSMTSF            0.55
1STFLRSF               0.54
TOTRMSABVGRD           0.53
GARAGETYPE_DETCHD      0.49
OPENPORCHSF            0.48
HEATINGQC              0.44
FIREPLACES             0.43
MASVNRAREA             0.42
FOUNDATION_CBLOCK      0.40
EXTERIOR1ST_VINYLSD    0.36
SALETYPE_NEW           0.30
EXTERIOR1ST_WD SDNG    0.22
EXTERIOR1ST_METALSD    0.21
SALETYPE_WD            0.19
GARAGETYPE_OTHER       0.16
EXTERIOR1ST_OTHER      0.03
SALETYPE_CON           0.02
EXTERIOR1ST_PLYWOOD    0.00
Name: SALEPRICE, dtype: float64